In [1]:
import pandas as pd
import ast

In [2]:
df = pd.read_csv("../../data/raw/movies_metadata.csv", low_memory=False)
df.shape

(45466, 24)

In [3]:
df[[
    "id",
    "original_title",
    "release_date",
    "genres",
    "production_countries",
    "spoken_languages"
]].head()

,id,original_title,release_date,genres,production_countries,spoken_languages
0,862,Toy Story,1995-10-30,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...","[{'iso_3166_1': 'US', 'name': 'United States o...","[{'iso_639_1': 'en', 'name': 'English'}]"
1,8844,Jumanji,1995-12-15,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...","[{'iso_3166_1': 'US', 'name': 'United States o...","[{'iso_639_1': 'en', 'name': 'English'}, {'iso..."
2,15602,Grumpier Old Men,1995-12-22,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...","[{'iso_3166_1': 'US', 'name': 'United States o...","[{'iso_639_1': 'en', 'name': 'English'}]"
3,31357,Waiting to Exhale,1995-12-22,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...","[{'iso_3166_1': 'US', 'name': 'United States o...","[{'iso_639_1': 'en', 'name': 'English'}]"
4,11862,Father of the Bride Part II,1995-02-10,"[{'id': 35, 'name': 'Comedy'}]","[{'iso_3166_1': 'US', 'name': 'United States o...","[{'iso_639_1': 'en', 'name': 'English'}]"


In [4]:
df[[
    "id",
    "original_title",
    "genres",
    "production_countries",
    "spoken_languages"
    ]].isna().sum()

id                      0
original_title          0
genres                  0
production_countries    3
spoken_languages        6
dtype: int64

In [5]:
nested_columns = [
    "genres",
    "production_countries",
    "spoken_languages"
]

for column in nested_columns:
    print(column, (df[column] == "[]").sum())

genres 2442
production_countries 6282
spoken_languages 3829


In [6]:
def check_nested_value(value):
    if pd.isna(value):
        return "missing"
    try:
        parsed = ast.literal_eval(value)
    except (ValueError, SyntaxError, TypeError):
        return "malformed"
    
    if parsed == []:
        return "empty_list"
    
    if not isinstance(parsed, list):
        return "not_a_list"
    
    return "valid"

In [7]:
for column in nested_columns:
    print(column)
    print(df[column].apply(check_nested_value).value_counts())
    print()

genres
genres
valid         43024
empty_list     2442
Name: count, dtype: int64

production_countries
production_countries
valid         39178
empty_list     6282
missing           3
not_a_list        3
Name: count, dtype: int64

spoken_languages
spoken_languages
valid         41631
empty_list     3829
missing           6
Name: count, dtype: int64



In [8]:
country_status = df["production_countries"].apply(check_nested_value)

df.loc[
    country_status == "not_a_list",
    ["id", "original_title", "production_countries"]
]

,id,original_title,production_countries
19730,1997-08-20,"[{'iso_639_1': 'en', 'name': 'English'}]",6.0
29503,2012-09-29,"[{'iso_639_1': 'ja', 'name': '日本語'}]",7.0
35587,2014-01-01,"[{'iso_639_1': 'en', 'name': 'English'}]",4.3


In [9]:
pd.set_option("display.max_columns", None)

df.loc[[19730, 29503, 35587]]

,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,popularity,poster_path,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
19730,- Written by Ørnås,0.065736,/ff9qCepilowshEtG2GYWwzt2bs4.jpg,"[{'name': 'Carousel Productions', 'id': 11176}...","[{'iso_3166_1': 'CA', 'name': 'Canada'}, {'iso...",1997-08-20,0,104.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Midnight Man,False,6.0,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
29503,Rune Balot goes to a casino connected to the ...,1.931659,/zV8bHuSL6WXoD6FWogP9j4x80bL.jpg,"[{'name': 'Aniplex', 'id': 2883}, {'name': 'Go...","[{'iso_3166_1': 'US', 'name': 'United States o...",2012-09-29,0,68.0,"[{'iso_639_1': 'ja', 'name': '日本語'}]",Released,NaN,Mardock Scramble: The Third Exhaust,False,7.0,12,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
35587,Avalanche Sharks tells the story of a bikini ...,2.185485,/zaSf5OG7V8X8gqFvly88zDdRm46.jpg,"[{'name': 'Odyssey Media', 'id': 17161}, {'nam...","[{'iso_3166_1': 'CA', 'name': 'Canada'}]",2014-01-01,0,82.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Beware Of Frost Bites,Avalanche Sharks,False,4.3,22,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
def check_inner_structure(value):
    if pd.isna(value):
        return "missing"

    try:
        parsed = ast.literal_eval(value)
    except (ValueError, SyntaxError, TypeError):
        return "malformed"

    if parsed == []:
        return "empty_list"

    if not isinstance(parsed, list):
        return "not_a_list"

    for item in parsed:
        if not isinstance(item, dict):
            return "item_not_dictionary"

        if "name" not in item:
            return "missing_name"

        if not isinstance(item["name"], str) or item["name"].strip() == "":
            return "empty_name"

    return "valid"

In [11]:
for column in nested_columns:
    print(column)
    print(df[column].apply(check_inner_structure).value_counts())
    print()

genres
genres
valid         43024
empty_list     2442
Name: count, dtype: int64

production_countries
production_countries
valid         39178
empty_list     6282
missing           3
not_a_list        3
Name: count, dtype: int64

spoken_languages
spoken_languages
valid         41147
empty_list     3829
empty_name      484
missing           6
Name: count, dtype: int64



In [12]:
language_status = df["spoken_languages"].apply(check_inner_structure)

df.loc[
    language_status == "empty_name",
    ["id", "original_title", "spoken_languages"]
].head(10)

,id,original_title,spoken_languages
108,197,Braveheart,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso..."
151,649,Belle de jour,"[{'iso_639_1': 'fr', 'name': 'Français'}, {'is..."
211,19155,Before the Rain,"[{'iso_639_1': 'sq', 'name': 'shqip'}, {'iso_6..."
262,4476,Legends of the Fall,"[{'iso_639_1': 'kw', 'name': ''}, {'iso_639_1'..."
284,241,Natural Born Killers,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso..."
286,527,Once Were Warriors,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso..."
297,30304,Picture Bride,"[{'iso_639_1': 'tl', 'name': ''}, {'iso_639_1'..."
413,46924,Being Human,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso..."
440,26141,Fatal Instinct,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso..."
504,713,The Piano,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso..."


In [13]:
def extract_names(value):
    if pd.isna(value):
        return pd.NA

    try:
        parsed = ast.literal_eval(value)
    except (ValueError, SyntaxError, TypeError):
        return pd.NA

    if not isinstance(parsed, list) or not parsed:
        return pd.NA

    names = []

    for item in parsed:
        if isinstance(item, dict):
            name = item.get("name")

            if isinstance(name, str):
                name = name.strip()

                if name and name not in names:
                    names.append(name)

    if not names:
        return pd.NA

    return "|".join(names)

## belongs_to_collection


In [14]:
test = df["belongs_to_collection"].iloc[0]
print(test)
print(type(test))

{'id': 10194, 'name': 'Toy Story Collection', 'poster_path': '/7G9915LfUQ2lVfwMEEhDsn3kT4B.jpg', 'backdrop_path': '/9FBwqcd9IRruEDUrTdcaafOMKUq.jpg'}
<class 'str'>


Convert to dictionary using `ast.literal_eval()`, extract the name of collection, then apply it in place. If the cell containted invalid data (ex. not a string), replace it with NA. Some cells had numerical values and nothing could be extracted; therefore, setting to NA makes the most sense.


In [15]:
for i in df["belongs_to_collection"].index:
    string = df['belongs_to_collection'].iloc[i]
    if pd.notna(string):
        conversion = ast.literal_eval(string)
        if isinstance(conversion, dict):
            df.at[i, 'belongs_to_collection'] = conversion["name"]
        else:
            df.at[i, 'belongs_to_collection'] = pd.NA


df["belongs_to_collection"].head()


0              Toy Story Collection
1                               NaN
2         Grumpy Old Men Collection
3                               NaN
4    Father of the Bride Collection
Name: belongs_to_collection, dtype: object

# production_companies


In [16]:
test = df["production_companies"].iloc[1]
print(test)
print(type(test))

[{'name': 'TriStar Pictures', 'id': 559}, {'name': 'Teitler Film', 'id': 2550}, {'name': 'Interscope Communications', 'id': 10201}]
<class 'str'>


A similar approach can be used as belongs_to_collection. However, it must be adapted since we now have a list of dictionaries, rather than just a dictionary.


In [17]:
df["production_companies"] = df["production_companies"].astype(object)

for i in df["production_companies"].index:
    string = df['production_companies'].iloc[i]
    if pd.notna(string):
        conversion = ast.literal_eval(string)
        if isinstance(conversion, list):
            lst = []
            for element in conversion:
                if element['name'] not in lst:
                    lst.append(element['name'])
            df.at[i, "production_companies"] = "|".join(lst)
        else:
            df.at[i, 'production_companies'] = pd.NA
            
df["production_companies"].head()


0                              Pixar Animation Studios
1    TriStar Pictures|Teitler Film|Interscope Commu...
2                          Warner Bros.|Lancaster Gate
3               Twentieth Century Fox Film Corporation
4            Sandollar Productions|Touchstone Pictures
Name: production_companies, dtype: object

In [18]:
df['release_date'].head()

0    1995-10-30
1    1995-12-15
2    1995-12-22
3    1995-12-22
4    1995-02-10
Name: release_date, dtype: object

In [19]:
converted_dates = pd.to_datetime(df["release_date"], errors="coerce")
print(df[converted_dates.isna()][['original_title', 'release_date']])
df['release_date'] = converted_dates

                                       original_title release_date
711              War Stories Our Mother Never Told Us          NaN
734                             Vermont Is for Lovers          NaN
3460                       Jails, Hospitals & Hip-Hop          NaN
3628                                   Boricua's Bond          NaN
5879                              Divine Intervention          NaN
...                                               ...          ...
45148               Konstruktor krasnogo tsveta -1993          NaN
45203  All Superheroes Must Die 2: The Last Superhero          NaN
45338                  The Land Where the Blues Began          NaN
45410                                           Aprel          NaN
45461                                         رگ خواب          NaN

[90 rows x 2 columns]


In [20]:
cleaned_df = df[[
    "id",
    "original_title",
    "release_date",
    "genres",
    "belongs_to_collection",
    "production_companies",
    "production_countries",
    "spoken_languages"
]].copy()

for column in nested_columns:
    cleaned_df[column] = cleaned_df[column].apply(extract_names)

cleaned_df.head()

,id,original_title,release_date,genres,belongs_to_collection,production_companies,production_countries,spoken_languages
0,862,Toy Story,1995-10-30,Animation|Comedy|Family,Toy Story Collection,Pixar Animation Studios,United States of America,English
1,8844,Jumanji,1995-12-15,Adventure|Fantasy|Family,NaN,TriStar Pictures|Teitler Film|Interscope Commu...,United States of America,English|Français
2,15602,Grumpier Old Men,1995-12-22,Romance|Comedy,Grumpy Old Men Collection,Warner Bros.|Lancaster Gate,United States of America,English
3,31357,Waiting to Exhale,1995-12-22,Comedy|Drama|Romance,NaN,Twentieth Century Fox Film Corporation,United States of America,English
4,11862,Father of the Bride Part II,1995-02-10,Comedy,Father of the Bride Collection,Sandollar Productions|Touchstone Pictures,United States of America,English


In [21]:
numeric_ids = pd.to_numeric(df["id"], errors="coerce")

invalid_ids = df[numeric_ids.isna()][
    ["id", "original_title"]
]

invalid_ids

,id,original_title
19730,1997-08-20,"[{'iso_639_1': 'en', 'name': 'English'}]"
29503,2012-09-29,"[{'iso_639_1': 'ja', 'name': '日本語'}]"
35587,2014-01-01,"[{'iso_639_1': 'en', 'name': 'English'}]"


In [22]:
df[df["id"].duplicated(keep=False)][
    ["id", "original_title"]
].sort_values("id")

,id,original_title
676,105045,Das Versprechen
1465,105045,Das Versprechen
44821,10991,Pokémon 3: The Movie
4114,10991,Pokémon 3: The Movie
5710,109962,Rich and Famous
20899,109962,Rich and Famous
23534,110428,Camille Claudel 1915
4356,110428,Camille Claudel 1915
24844,11115,Deal
14012,11115,Deal


In [23]:
print("Duplicate rows:", df["id"].duplicated(keep=False).sum())
print("Unique duplicated IDs:", df.loc[df["id"].duplicated(keep=False), "id"].nunique())

Duplicate rows: 59
Unique duplicated IDs: 29


In [24]:
duplicate_rows = cleaned_df[
    cleaned_df["id"].duplicated(keep=False)
].sort_values("id")

duplicate_rows

,id,original_title,release_date,genres,belongs_to_collection,production_companies,production_countries,spoken_languages
676,105045,Das Versprechen,1995-02-16,Drama|Romance,NaN,Studio Babelsberg|Centre National de la Cinéma...,Germany,Deutsch
1465,105045,Das Versprechen,1995-02-16,Drama|Romance,NaN,Studio Babelsberg|Centre National de la Cinéma...,Germany,Deutsch
44821,10991,Pokémon 3: The Movie,2000-07-08,Adventure|Fantasy|Animation|Action|Family,Pokémon Collection,TV Tokyo|4 Kids Entertainment|Nintendo|Pikachu...,Japan,English
4114,10991,Pokémon 3: The Movie,2000-07-08,Adventure|Fantasy|Animation|Action|Family,Pokémon Collection,TV Tokyo|4 Kids Entertainment|Nintendo|Pikachu...,Japan,English
5710,109962,Rich and Famous,1981-09-23,Drama,NaN,Metro-Goldwyn-Mayer (MGM)|Jaquet,United States of America,English
20899,109962,Rich and Famous,1981-09-23,Drama,NaN,Metro-Goldwyn-Mayer (MGM)|Jaquet,United States of America,English
23534,110428,Camille Claudel 1915,2013-03-13,Drama,NaN,Canal+|Arte France Cinéma|3B Productions|C.R.R...,France,Français
4356,110428,Camille Claudel 1915,2013-03-13,Drama,NaN,Canal+|Arte France Cinéma|3B Productions|C.R.R...,France,Français
24844,11115,Deal,2008-01-29,Comedy|Drama,NaN,Andertainment Group|Crescent City Pictures|Tag...,United States of America,English
14012,11115,Deal,2008-01-29,Comedy|Drama,NaN,Andertainment Group|Crescent City Pictures|Tag...,United States of America,English


In [25]:
duplicate_check = (
    cleaned_df
    .groupby("id", dropna=False)
    .nunique(dropna=False)
)

duplicate_check[
    (duplicate_check > 1).any(axis=1)
]

,original_title,release_date,genres,belongs_to_collection,production_companies,production_countries,spoken_languages
id,,,,,,,


In [26]:
cleaned_df = cleaned_df.drop_duplicates(subset="id", keep="first")

cleaned_df.shape

(45436, 8)

In [27]:
cleaned_df = cleaned_df[
    pd.to_numeric(cleaned_df["id"], errors="coerce").notna()
].copy()

cleaned_df.shape

(45433, 8)

In [28]:
blank_titles = cleaned_df[
    cleaned_df["original_title"].astype(str).str.strip() == ""
]

blank_titles

,id,original_title,release_date,genres,belongs_to_collection,production_companies,production_countries,spoken_languages


In [29]:
cleaned_df[[
    "genres",
    "production_countries",
    "spoken_languages"
]].isna().sum()

genres                  2442
production_countries    6282
spoken_languages        3955
dtype: int64

In [30]:
def has_duplicate_names(value):
    if pd.isna(value):
        return False

    names = value.split("|")
    return len(names) != len(set(names))

for column in nested_columns:
    print(column, cleaned_df[column].apply(has_duplicate_names).sum())

genres 0
production_countries 0
spoken_languages 0


In [ ]:
cleaned_df["id"].duplicated().sum()

np.int64(0)

In [35]:
for column in nested_columns:
    print(column, (cleaned_df[column] == "").sum())

genres 0
production_countries 0
spoken_languages 0


In [34]:
# save cleaned_df
cleaned_df.to_csv("../../data/processed/nested_metadata.csv", index=False)